# Parquet - Missing Values

Introduce seeded missing field values without removing rows or changing other fields.

**Instructions:** Run `prepare_data_parquet.ipynb` to create clean `data/weather.parquet` first (`uv add pyarrow`). This notebook overwrites the Parquet file; restore the clean artifact before rerunning. For combination, use only the required config/function/call, and load/save once.

In [ ]:
from pathlib import Path
from random import Random

import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

# Configuration
DATA_PATH = Path("data/weather.parquet")
TIME_COLUMN = "time"
TIMEZONE = "America/Edmonton"
START_DATE = "2025-08-01"
END_DATE = "2025-09-01"  # Exclusive; local dates.
SEED = 814205

# Missing Values
MISSING_TARGET_COLUMN = "relative_humidity_2m"
MISSING_METHOD = "mixed"  # scattered, grouped, mixed, random
MISSING_RATE = (0.03, 0.08)  # Share of current eligible nonmissing values changed.
MISSING_GROUPS = 3
MISSING_GROUPED_SHARE = 0.70

In [ ]:
# Shared selection: include once in an assembled dirty_data.py.
def scoped_rows(df, *, time_column, timezone, start_date, end_date):
    if time_column not in df:
        raise ValueError("Time column not found.")
    times = pd.to_datetime(df[time_column], utc=True, errors="raise")
    if times.isna().any():
        raise ValueError("Time column contains missing timestamps.")
    start = pd.Timestamp(start_date, tz=timezone).tz_convert("UTC")
    end = pd.Timestamp(end_date, tz=timezone).tz_convert("UTC")
    if start >= end:
        raise ValueError("Start date must precede end date.")
    scoped = df.loc[times.ge(start) & times.lt(end)]
    if scoped.empty:
        raise ValueError("No records in the selected interval.")
    # Indices refer to the input DataFrame. No complete-hourly assumption.
    return scoped.sort_values(time_column, kind="stable").index.tolist(), start, end

def select_positions(n, rng, *, method, rate, n_groups, grouped_share):
    """Select a fraction of the current eligible records in ordered positions.

    Grouped blocks have at least two consecutive eligible records; mixed
    placement reserves at least one independently scattered selection.
    """
    if n < 1:
        raise ValueError("No eligible records.")
    if method not in ("scattered", "grouped", "mixed", "random"):
        raise ValueError("Unknown placement method.")
    if not 0 < rate[0] <= rate[1] < 1:
        raise ValueError("Rate must be between 0 and 1.")
    if n_groups < 1 or not 0 < grouped_share < 1:
        raise ValueError("Invalid group count or grouped share.")

    chosen = rng.choice(("scattered", "grouped", "mixed")) if method == "random" else method
    count = max(1, round(n * rng.uniform(*rate)))
    if chosen == "mixed" and count < 3:
        raise ValueError("Mixed placement needs two grouped and one scattered record.")
    grouped_count = (count if chosen == "grouped" else
                     max(2, min(count - 1, round(count * grouped_share))) if chosen == "mixed" else 0)
    selected = set()
    if grouped_count:
        if grouped_count < 2:
            raise ValueError("Grouped placement requires at least two affected records.")
        groups = min(n_groups, grouped_count // 2)
        if n - grouped_count < groups - 1:
            raise ValueError("Insufficient eligible records for separate groups.")
        length, extra = divmod(grouped_count, groups)
        lengths = [length + (i < extra) for i in range(groups)]
        rng.shuffle(lengths)
        slack = n - grouped_count - (groups - 1)
        dividers = sorted(rng.sample(range(slack + groups), groups))
        gaps = ([dividers[0]] +
                [dividers[i] - dividers[i - 1] - 1 for i in range(1, groups)] +
                [slack + groups - 1 - dividers[-1]])
        cursor = gaps[0]
        for i, block_length in enumerate(lengths):
            selected.update(range(cursor, cursor + block_length))
            if i < groups - 1:
                cursor += block_length + 1 + gaps[i + 1]

    scattered_count = count - grouped_count
    if scattered_count:
        remaining = [i for i in range(n) if i not in selected]
        selected.update(rng.sample(remaining, k=scattered_count))
    if len(selected) != count:
        raise AssertionError("Selected count differs from the intended count.")
    return sorted(selected), chosen


In [ ]:
def add_missing_values(
    df, rng, *, time_column, timezone, start_date, end_date,
    target_column, method, missing_rate, n_groups, grouped_share,
):
    if target_column not in df or target_column == time_column:
        raise ValueError("Select an existing non-time target column.")
    dirty = df.reset_index(drop=True).copy()
    eligible, _, _ = scoped_rows(
        dirty, time_column=time_column, timezone=timezone,
        start_date=start_date, end_date=end_date,
    )
    if dirty.loc[eligible, target_column].isna().any():
        raise ValueError("Target already contains nulls in scope; restore clean data first.")
    selected, _ = select_positions(
        len(eligible), rng, method=method, rate=missing_rate,
        n_groups=n_groups, grouped_share=grouped_share,
    )
    if pd.api.types.is_integer_dtype(dirty[target_column]):
        dirty[target_column] = dirty[target_column].astype("Int64")
    elif pd.api.types.is_bool_dtype(dirty[target_column]):
        dirty[target_column] = dirty[target_column].astype("boolean")
    dirty.loc[[eligible[i] for i in selected], target_column] = pd.NA
    return dirty

In [ ]:
# Small controlled trial (no source file changed).
trial_times = pd.date_range("2025-08-01", periods=7*24, freq="h", tz=TIMEZONE).tz_convert("UTC")
trial = pd.DataFrame({
    "time": trial_times,
    "temperature_2m": [float(10 + i % 17) for i in range(len(trial_times))],
    "relative_humidity_2m": pd.Series([int(40 + i % 40) for i in range(len(trial_times))], dtype="int64"),
    "precipitation": [float(i % 10) for i in range(len(trial_times))],
})

trial_dirty = add_missing_values(trial, Random(SEED),
    time_column="time", timezone=TIMEZONE, start_date="2025-08-01", end_date="2025-08-08",
    target_column="relative_humidity_2m", method="mixed", missing_rate=(0.10, 0.10),
    n_groups=2, grouped_share=0.70)
assert len(trial_dirty) == len(trial)
assert trial_dirty["relative_humidity_2m"].isna().sum() == round(len(trial) * 0.10)
assert str(trial_dirty["relative_humidity_2m"].dtype) == "Int64"
pd.testing.assert_frame_equal(trial_dirty.drop(columns="relative_humidity_2m"),
                              trial.drop(columns="relative_humidity_2m"))
trial_dirty.loc[trial_dirty["relative_humidity_2m"].isna()].head()

In [ ]:
# Verify actual Parquet types and values on a small temporary file.
from tempfile import TemporaryDirectory
with TemporaryDirectory() as temporary:
    clean_path = Path(temporary) / "clean.parquet"
    temp_path = Path(temporary) / "trial.parquet"
    trial.to_parquet(clean_path, index=False)
    trial_dirty.to_parquet(temp_path, index=False)
    roundtrip = pd.read_parquet(temp_path)
    before_schema = pq.read_schema(clean_path)
    after_schema = pq.read_schema(temp_path)
    assert len(roundtrip) == len(trial_dirty)
    assert roundtrip.columns.tolist() == trial_dirty.columns.tolist()
    assert pq.read_metadata(temp_path).num_rows == len(trial_dirty)
    assert pq.read_table(temp_path)["relative_humidity_2m"].null_count > 0
    assert before_schema.field("relative_humidity_2m").type == after_schema.field("relative_humidity_2m").type
    print("Stored Parquet schema:", after_schema)

In [ ]:
# Apply the reusable function to a freshly prepared Parquet artifact.
clean = pd.read_parquet(DATA_PATH)
rng = Random(SEED)
dirty = add_missing_values(
    clean, rng, time_column=TIME_COLUMN, timezone=TIMEZONE,
    start_date=START_DATE, end_date=END_DATE,
    target_column=MISSING_TARGET_COLUMN, method=MISSING_METHOD,
    missing_rate=MISSING_RATE, n_groups=MISSING_GROUPS,
    grouped_share=MISSING_GROUPED_SHARE,
)
dirty.to_parquet(DATA_PATH, index=False)
saved = pd.read_parquet(DATA_PATH)

In [ ]:
# Inspect: missing cells (including nullable integer values).
saved.isna().any().any(), saved.isna().sum()

In [ ]:
# Verify persistence: all changes exist after writing the Parquet file.
assert len(saved) == len(dirty)
assert saved.columns.tolist() == dirty.columns.tolist()
assert pq.read_metadata(DATA_PATH).num_rows == len(saved)
assert saved[MISSING_TARGET_COLUMN].isna().sum() - clean[MISSING_TARGET_COLUMN].isna().sum() > 0
pd.testing.assert_frame_equal(saved.drop(columns=MISSING_TARGET_COLUMN),
                              clean.drop(columns=MISSING_TARGET_COLUMN), check_dtype=False)
print("New null values:", int(saved[MISSING_TARGET_COLUMN].isna().sum() - clean[MISSING_TARGET_COLUMN].isna().sum()))

In [ ]:
# Daily count of newly missing values.
dates = pd.to_datetime(saved[TIME_COLUMN], utc=True).dt.tz_convert(TIMEZONE).dt.date
counts = saved[MISSING_TARGET_COLUMN].isna().groupby(dates).sum()
fig, ax = plt.subplots(figsize=(9, 3))
counts.loc[lambda s: s.gt(0)].plot.bar(ax=ax)
ax.set(title="Days with missing measurements", xlabel="Local date", ylabel="Missing values")
plt.tight_layout(); plt.show()